In [2]:
# Pipeline
# Step 1 of Pipeline: Loading & Inspecting Raw Data

In [1]:
import pandas as pd
import sqlite3

# 1. Load raw datasets
df_clients = pd.read_csv("dim_clients.csv")
df_trades = pd.read_csv("fact_trades.csv")
df_cx = pd.read_csv("fact_cx.csv")

# 2. Inspect Data Shapes & Missing Values
print("--- DATASET OVERVIEW ---")
print(f"Clients: {df_clients.shape[0]} rows, {df_clients.shape[1]} columns")
print(f"Trades:  {df_trades.shape[0]} rows, {df_trades.shape[1]} columns")
print(f"CX Logs: {df_cx.shape[0]} rows, {df_cx.shape[1]} columns\n")

# Check data types
print("--- TRADES SCHEMA ---")
print(df_trades.dtypes)

--- DATASET OVERVIEW ---
Clients: 150 rows, 4 columns
Trades:  10000 rows, 10 columns
CX Logs: 2500 rows, 6 columns

--- TRADES SCHEMA ---
trade_id                     str
trade_date                   str
client_id                    str
asset_class                  str
notional_amount          float64
internal_clearing_fee    float64
custodian_charged_fee    float64
settlement_delay_days      int64
trade_failed               int64
fail_reason                  str
dtype: object


In [ ]:
# Step 2 of Pipeline: SQL Database Staging

In [3]:
# Create local SQLite database connection
conn = sqlite3.connect("investment_bank_ops.db")

# Write DataFrames to SQL tables
df_clients.to_sql("dim_clients", conn, if_exists="replace", index=False)
df_trades.to_sql("fact_trades", conn, if_exists="replace", index=False)
df_cx.to_sql("fact_cx", conn, if_exists="replace", index=False)

print("\nSuccessfully staged tables into SQLite database: 'investment_bank_ops.db'")


Successfully staged tables into SQLite database: 'investment_bank_ops.db'


In [5]:
import pandas as pd
import numpy as np
import sqlite3

# -------------------------------------------------------------------
# STEP 1: LOAD RAW CSVS INTO PANDAS
# -------------------------------------------------------------------
df_clients = pd.read_csv("dim_clients.csv")
df_trades = pd.read_csv("fact_trades.csv")
df_cx = pd.read_csv("fact_cx.csv")

print("Data Loaded Successfully.")
print(f"Clients shape: {df_clients.shape}")
print(f"Trades shape:  {df_trades.shape}")
print(f"CX Logs shape: {df_cx.shape}")

# -------------------------------------------------------------------
# STEP 2: DATA CLEANING & TYPE CONVERSIONS
# -------------------------------------------------------------------
# Ensure trade_date is proper datetime format
df_trades['trade_date'] = pd.to_datetime(df_trades['trade_date'], format='%d-%m-%Y')

# Handle missing or invalid values if any
df_trades['fail_reason'] = df_trades['fail_reason'].fillna('Unknown')
df_cx['csat_score'] = df_cx['csat_score'].fillna(0)

# -------------------------------------------------------------------
# STEP 3: FEATURE ENGINEERING (Pipeline Enhancements)
# -------------------------------------------------------------------
# 1. Calculate Fee Discrepancy Amount & Flag (eClerx Scenario Prep)
df_trades['fee_discrepancy'] = np.round(
    df_trades['custodian_charged_fee'] - df_trades['internal_clearing_fee'], 2
)
df_trades['is_fee_mismatch'] = np.where(df_trades['fee_discrepancy'] > 0, 1, 0)

# 2. Add Settlement SLA Flag
df_trades['sla_breach_flag'] = np.where(df_trades['settlement_delay_days'] > 2, 1, 0)

# -------------------------------------------------------------------
# STEP 4: STAGE TABLES INTO SQLITE DATABASE
# -------------------------------------------------------------------
# Connects to or creates the local SQL database file
conn = sqlite3.connect("investment_bank_ops.db")

# Write engineered tables into SQLite
df_clients.to_sql("dim_clients", conn, if_exists="replace", index=False)
df_trades.to_sql("fact_trades", conn, if_exists="replace", index=False)
df_cx.to_sql("fact_cx", conn, if_exists="replace", index=False)

print("\n--- PIPELINE STATUS ---")
print("1. Data cleaned and verified.")
print("2. Derived pipeline features created (fee_discrepancy, sla_breach_flag).")
print("3. Staged into SQLite database file: 'investment_bank_ops.db'")

# -------------------------------------------------------------------
# VERIFY WITH A QUICK SQL QUERY
# -------------------------------------------------------------------
verification_query = """
SELECT 
    t.asset_class,
    COUNT(t.trade_id) AS total_trades,
    SUM(t.trade_failed) AS total_failed,
    ROUND(AVG(t.fee_discrepancy), 2) AS avg_fee_discrepancy
FROM fact_trades t
GROUP BY t.asset_class;
"""

df_summary = pd.read_sql_query(verification_query, conn)
print("\nSQL Verification Check:")
print(df_summary)

conn.close()

Data Loaded Successfully.
Clients shape: (150, 4)
Trades shape:  (10000, 10)
CX Logs shape: (2500, 6)

--- PIPELINE STATUS ---
1. Data cleaned and verified.
2. Derived pipeline features created (fee_discrepancy, sla_breach_flag).
3. Staged into SQLite database file: 'investment_bank_ops.db'

SQL Verification Check:
       asset_class  total_trades  total_failed  avg_fee_discrepancy
0         Equities          3988           559                 3.62
1       FX Options           973           122                 3.20
2     Fixed Income          3051           457                 2.62
3  OTC Derivatives          1988           488                 3.63


In [ ]:
# Scenario 1: Early Warning System for Trade Failures

# Identify High -risk failing trades (trade_failed = 1) based on operational indicators available at trade execution:

# asset_class: Complex instruments like OTC Derivatives have higher failure risk than Equities.
# notional_amount: High-value trades often face additional clearing or liquidity checks.
# settlement_delay_days: Historical processing latency for the trade flow.
# client_tier: Tier 1 vs. Tier 2 vs. HNW operational profiles.

In [8]:
import pandas as pd
import numpy as np
import sqlite3
import statsmodels.api as sm

# 1. Connect to SQLite and fetch joined trade & client data
conn = sqlite3.connect("investment_bank_ops.db")

query = """
SELECT 
    t.trade_id,
    t.client_id,
    c.client_tier,
    t.asset_class,
    t.notional_amount,
    t.settlement_delay_days,
    t.trade_failed
FROM fact_trades t
JOIN dim_clients c ON t.client_id = c.client_id
"""

df = pd.read_sql_query(query, conn)

# 2. Prepare feature set for Logistic Regression
# Convert categorical features (asset_class, client_tier) into dummy variables
df_model = pd.get_dummies(df[['asset_class', 'client_tier', 'notional_amount', 'settlement_delay_days']], drop_first=True)

# Define X (features) and y (target variable: trade_failed)
X = sm.add_constant(df_model.astype(float))
y = df['trade_failed']

# 3. Fit Logistic Regression Model
logit_model = sm.Logit(y, X).fit()

# Print summary to verify statistical significance (p-values, odds ratios)
print("--- MODEL SUMMARY ---")
print(logit_model.summary())

# 4. Generate Fail Probability Scores & Risk Flags
df['fail_probability'] = np.round(logit_model.predict(X), 4)

# Define Early Warning Alert threshold (e.g., probability > 35% = High Risk Alert)
df['risk_alert_flag'] = np.where(df['fail_probability'] > 0.35, 'HIGH RISK - Early Warning', 'Normal')

# 5. Update SQLite database with the new predictive outputs
# Save predictions into a new table or update existing schema
df[['trade_id', 'fail_probability', 'risk_alert_flag']].to_sql("pred_trade_risk", conn, if_exists="replace", index=False)

print("\n--- SCENARIO 1 COMPLETE ---")
print(f"Total High-Risk Trades Flagged: {(df['risk_alert_flag'] == 'HIGH RISK - Early Warning').sum()}")
print("Results saved to SQL table: 'pred_trade_risk'")

conn.close()

Optimization terminated successfully.
         Current function value: 0.412645
         Iterations 6
--- MODEL SUMMARY ---
                           Logit Regression Results                           
Dep. Variable:           trade_failed   No. Observations:                10000
Model:                          Logit   Df Residuals:                     9992
Method:                           MLE   Df Model:                            7
Date:                Fri, 25 Sep 2026   Pseudo R-squ.:                 0.07053
Time:                        16:23:11   Log-Likelihood:                -4126.4
converged:                       True   LL-Null:                       -4439.6
Covariance Type:            nonrobust   LLR p-value:                5.462e-131
                                       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
const                               -2.565

In [9]:
#pip install statsmodels

In [ ]:
# Key Takeaways from the Model Output:

# 1. Statistical Drivers ($P > \vert{}z\vert{} < 0.05$):
# settlement_delay_days ($p = 0.000$, coef = $0.5308$): Every additional day of settlement delay significantly 
# increases the log-odds of a trade failing. This is a primary driver.

# asset_class_OTC Derivatives ($p = 0.000$, coef = $0.6880$): 
# OTC Derivatives are significantly more prone to failure compared to the baseline (Equities), reflecting the 
# complex nature of confirmation and margin checks for derivatives.

# 2. Non-Significant Variables:
# notional_amount ($p = 0.249$) and client_tier ($p > 0.4$) do not show a statistically significant direct impact
# on failure probability in this run.

# 3. Business Impact:
# 528 trades out of 10,000 were flagged as HIGH RISK - Early Warning (probability $> 35\%$).
# By saving pred_trade_risk into investment_bank_ops.db, your CX team now has an actionable queue to prioritize 
# high-risk trades before settlement day.

In [ ]:
# Scenario 2: CSAT & MTTR Impact Analysis (MTTR - mean time to resolution)
# Analytic Technique: Inferential Statistics (Hypothesis Testing & Linear Regression)

# How much does resolution time (MTTR) actually harm our client satisfaction scores across different issue types?

In [10]:
import pandas as pd
import sqlite3
import statsmodels.api as sm

# 1. Connect to SQLite database
conn = sqlite3.connect("investment_bank_ops.db")

# 2. Extract joined CX and Client data
query = """
SELECT 
    c.ticket_id,
    c.client_id,
    cl.client_tier,
    c.ticket_category,
    c.resolution_time_hrs,
    c.csat_score
FROM fact_cx c
JOIN dim_clients cl ON c.client_id = cl.client_id
WHERE c.csat_score > 0
"""

df_cx = pd.read_sql_query(query, conn)

# 3. Fit OLS Regression to model CSAT Score
# Features: resolution_time_hrs and ticket_category
X = pd.get_dummies(df_cx[['resolution_time_hrs', 'ticket_category']], drop_first=True)
X = sm.add_constant(X.astype(float))
y = df_cx['csat_score']

ols_model = sm.OLS(y, X).fit()

print("--- SCENARIO 2: CSAT vs MTTR MODEL SUMMARY ---")
print(ols_model.summary())

# 4. Aggregate CX metrics per Client Tier for Power BI / Tableau consumption
query_agg = """
SELECT 
    cl.client_tier,
    c.ticket_category,
    COUNT(c.ticket_id) AS total_tickets,
    ROUND(AVG(c.resolution_time_hrs), 2) AS avg_mttr_hours,
    ROUND(AVG(c.csat_score), 2) AS avg_csat_score
FROM fact_cx c
JOIN dim_clients cl ON c.client_id = cl.client_id
GROUP BY cl.client_tier, c.ticket_category;
"""

df_cx_summary = pd.read_sql_query(query_agg, conn)

# Save summary metrics back to SQL
df_cx_summary.to_sql("agg_cx_performance", conn, if_exists="replace", index=False)

print("\n--- SCENARIO 2 COMPLETE ---")
print("CX Performance Summary saved to SQL table: 'agg_cx_performance'")

conn.close()

--- SCENARIO 2: CSAT vs MTTR MODEL SUMMARY ---
                            OLS Regression Results                            
Dep. Variable:             csat_score   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.164
Date:                Fri, 25 Sep 2026   Prob (F-statistic):              0.325
Time:                        19:13:27   Log-Likelihood:                -4077.6
No. Observations:                2500   AIC:                             8165.
Df Residuals:                    2495   BIC:                             8194.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------

In [ ]:
# In our initial random synthetic data generation script, CSAT scores were assigned randomly 
# (np.random.choice([1, 2, 3, 4, 5])) without anchoring them to resolution_time_hrs.

# To make Scenario 2 reflect real-world Bank of America operations, CSAT must logically decline as resolution 
# hours increase.

In [11]:
import pandas as pd
import numpy as np
import sqlite3
import statsmodels.api as sm

conn = sqlite3.connect("investment_bank_ops.db")

# 1. Fetch existing CX data
df_cx = pd.read_sql_query("SELECT * FROM fact_cx", conn)

# 2. Inject realistic business logic:
# Base score 5.0 minus penalty for high resolution time and settlement delay issues
np.random.seed(42)
csat_calculated = (
    5.0 
    - (df_cx['resolution_time_hrs'] * 0.08) 
    - np.where(df_cx['ticket_category'] == 'Settlement Delay', 0.6, 0.0)
    + np.random.normal(0, 0.4, len(df_cx))
)

# Clip CSAT between 1 and 5 and round
df_cx['csat_score'] = np.clip(np.round(csat_calculated), 1, 5).astype(int)

# 3. Save updated CX table back to SQLite
df_cx.to_sql("fact_cx", conn, if_exists="replace", index=False)

# 4. Re-run OLS Model
X = pd.get_dummies(df_cx[['resolution_time_hrs', 'ticket_category']], drop_first=True)
X = sm.add_constant(X.astype(float))
y = df_cx['csat_score']

ols_model = sm.OLS(y, X).fit()

print("--- REVISED SCENARIO 2 MODEL SUMMARY ---")
print(ols_model.summary())

conn.close()

--- REVISED SCENARIO 2 MODEL SUMMARY ---
                            OLS Regression Results                            
Dep. Variable:             csat_score   R-squared:                       0.775
Model:                            OLS   Adj. R-squared:                  0.775
Method:                 Least Squares   F-statistic:                     2152.
Date:                Fri, 25 Sep 2026   Prob (F-statistic):               0.00
Time:                        19:18:41   Log-Likelihood:                -1737.4
No. Observations:                2500   AIC:                             3485.
Df Residuals:                    2495   BIC:                             3514.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------

In [ ]:
# 1. Overall Model Strength (R^2 = 0.775, p < 0.001)
# 77.5\% of the variation in CSAT scores is explained by resolution time (MTTR) and ticket category.
# The overall model is highly statistically significant (F\text{-statistic} = 2152, p = 0.000).

# 2. Key Statistical DriversResolution Time (resolution_time_hrs, p = 0.000, coef = -0.0736):
# Every 10 additional hours it takes to resolve a client ticket drops the CSAT score by ~0.74 points.
# This quantifies the exact operational cost of latency on client satisfaction.

# 3. ticket_category_Settlement Delay (p = 0.000, coef = -0.5510):
# Even holding resolution time constant, Settlement Delay tickets inherently suffer an automatic ~0.55 point 
# penalty in CSAT compared to standard queries. Clients are far more frustrated when trade settlement is at risk.

In [ ]:
# Scenario 3: Operational Health Scorecard Modeling
# Analytic Technique: Point-Based Scorecard Modeling (Index Calculation & Risk Binning)

In [ ]:
# a single, unified metric—a Client Operational Health Scorecard (0–100 scale)—that aggregates multiple risk 
# vectors per institutional client:

# Trade Failure Rate (%)
# Average Settlement Delay (Days)
# Average Resolution Time / MTTR (Hours)
# Average CSAT Score

In [12]:
import pandas as pd
import numpy as np
import sqlite3

# 1. Connect to SQLite database
conn = sqlite3.connect("investment_bank_ops.db")

# 2. Extract joined client-level operational metrics using SQL
sql_scorecard_input = """
SELECT 
    c.client_id,
    c.client_name,
    c.client_tier,
    c.region,
    COUNT(DISTINCT t.trade_id) AS total_trades,
    ROUND(AVG(t.trade_failed) * 100, 2) AS fail_rate_pct,
    ROUND(AVG(t.settlement_delay_days), 2) AS avg_delay_days,
    ROUND(AVG(cx.resolution_time_hrs), 2) AS avg_mttr_hrs,
    ROUND(AVG(cx.csat_score), 2) AS avg_csat
FROM dim_clients c
LEFT JOIN fact_trades t ON c.client_id = t.client_id
LEFT JOIN fact_cx cx ON c.client_id = cx.client_id
GROUP BY c.client_id, c.client_name, c.client_tier, c.region
"""

df_scorecard = pd.read_sql_query(sql_scorecard_input, conn)

# 3. Calculate Scorecard Components (Normalized 0-100 base scale)
# Penalize high fail rate, high delay, and high MTTR; Reward high CSAT
base_score = 100
fail_penalty = df_scorecard['fail_rate_pct'] * 1.5
delay_penalty = df_scorecard['avg_delay_days'] * 8.0
mttr_penalty = df_scorecard['avg_mttr_hrs'] * 1.2
csat_bonus = (df_scorecard['avg_csat'] - 3.0) * 10.0

raw_health_score = base_score - fail_penalty - delay_penalty - mttr_penalty + csat_bonus

# Clip score between 0 and 100
df_scorecard['health_score'] = np.round(np.clip(raw_health_score, 0, 100), 1)

# 4. Bin into Operational Health Risk Bands
conditions = [
    (df_scorecard['health_score'] >= 80),
    (df_scorecard['health_score'] >= 60) & (df_scorecard['health_score'] < 80),
    (df_scorecard['health_score'] < 60)
]
choices = ['Green - Healthy', 'Amber - Friction', 'Red - Critical Risk']

df_scorecard['health_band'] = np.select(conditions, choices, default='Unknown')

# 5. Output Scorecard Table to Database
df_scorecard.to_sql("dim_client_scorecard", conn, if_exists="replace", index=False)

print("--- SCENARIO 3: CLIENT HEALTH SCORECARD SUMMARY ---")
print(df_scorecard['health_band'].value_counts())
print("\nSample Client Health Records:")
print(df_scorecard[['client_id', 'client_tier', 'fail_rate_pct', 'avg_csat', 'health_score', 'health_band']].head(10))

conn.close()

--- SCENARIO 3: CLIENT HEALTH SCORECARD SUMMARY ---
health_band
Red - Critical Risk    84
Amber - Friction       65
Green - Healthy         1
Name: count, dtype: int64

Sample Client Health Records:
  client_id           client_tier  fail_rate_pct  avg_csat  health_score  \
0  CLI-1000  Institutional Tier 2          23.29      3.95          49.2   
1  CLI-1001            HNW Wealth          17.86      3.74          55.1   
2  CLI-1002            HNW Wealth          16.92      3.78          57.2   
3  CLI-1003  Institutional Tier 2          22.73      3.73          48.1   
4  CLI-1004  Institutional Tier 1          17.86      3.10          35.2   
5  CLI-1005  Institutional Tier 1          13.79      3.65          57.9   
6  CLI-1006  Institutional Tier 1          13.56      3.60          60.2   
7  CLI-1007            HNW Wealth          15.79      3.50          51.7   
8  CLI-1008  Institutional Tier 2          21.88      3.95          49.2   
9  CLI-1009            HNW Wealth        

In [ ]:
# Red - Critical Risk: 84 clients
# Amber - Friction: 65 clients
# Green - Healthy: 1 client

#Why is the Distribution Heavily Skewed to Red?
# In our formula, we set aggressive penalties:
# Health Score = 100 - (Fail Rate * 1.5) - (Delay Days * 8.0) - (MTTR Hours * 1.2) + (CSAT Bonus)

# Because average fail rates were around 15-20%, resolution times averaged 12-15 hours, and settlement delays 
# averaged 1-2 days, the cumulative penalties easily wiped out 40-50 points per client.

In [13]:
# Run this in your notebook to release the file lock:
try:
    conn.close()
    print("Database connection closed successfully!")
except:
    print("No active connection found.")

Database connection closed successfully!
